# Prepare input data for evapotranspiration modelling

In this notebook, data required to run evapotranspiration modelling, which was collected in the previous notebook, is pre-processed to be used as into to TSEB evapotranspiration model.

This notebook can be run on the [Copernicus Dataspace Jupyterhub](https://jupyterhub.dataspace.copernicus.eu) in which case data downloads are downloaded since both the data and the execution environment are on CDSE. 

**Note** You should select on of the kernels with GDAL installed, eg. "Geo science".

First we check that Sen-ET Toolbox is installed (and install it if necessary) and then we import all the necessary packages.

In [ ]:
try:
    import senet_toolbox
except ModuleNotFoundError:
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

In [ ]:
from pathlib import Path
from datetime import datetime
import re

from senet_toolbox import date_selector, visualization, raster_utils, read_area_date_info, load_lut
from senet_toolbox.workflows.decision_tree_sharpener import sharpen_lst
from senet_toolbox.workflows.prepare_ancillary_data import prepare_dem, prepare_lut_maps
from senet_toolbox.workflows.meteo_preprocessing import get_meteo_data
from senet_toolbox.workflows.biophysical_processing import biopar_biophysical_params

## 1. Select Area of Interest
To keep the data organised, and make processing of timeseries easier, the input and output data is kept in Area of Interest (AOI) folders. All data within an AOI folder has the same extent and grid. 

In the cell below, select the data location and AOI name which was set in input data collection notebook. It will also pick the last date which was processed by that notebook. If you would like to work with a different date then use the second cell to list the dates within the AOI for which input data was collected, select the one you want to work with from the drop down list and run the third cell to save your choice. 

In [ ]:
data_dir = "./mystorage/"
aoi_name = "botswana2"
aoi_data_dir = Path(data_dir) / aoi_name
date, bbox = read_area_date_info(dir=aoi_data_dir)
date = str(date)
date_data_dir = aoi_data_dir / date.replace("-", "")
s2_path = date_data_dir / f"s2_{date_data_dir.name}_data.nc"
worldcover_path = aoi_data_dir / "WorldCover2021.tif"
print(date)

In [ ]:
# Run this and next cells if you would like to work with a different date
date_selection = date_selector.get_collected_dates(
    aoi_data_dir = Path(data_dir) / aoi_name
)

In [ ]:
date = date_selection.value
date_data_dir = aoi_data_dir / date.replace("-", "")
s2_path = date_data_dir / f"s2_{date_data_dir.name}_data.nc"
worldcover_path = aoi_data_dir / "WorldCover2021.tif"

## 2. Prepare static input data

The input data derived from digital elevation model (DEM) and from the landcover map do not change between dates so this only needs to be done once when setting up a new AOI.

**Note:** By default the static parameters are set based on Worldcover landcover map and the [default look-up-table](https://github.com/DHI/Sen-ET-OpenEO-toolbox/blob/main/senet_toolbox/static_data/WorldCover10m_2020_LUT.csv). You can alo specify paths to your custom landcover maps and look-up-tables (e.g. covering specific crops). The columns of the custom look-up-table should be the same as for the default ones and the rows should match the classes present in the landcover map.

In [ ]:
prepare_dem(aoi_data_dir / "cdem.tif")

In [ ]:
lut = load_lut()
prepare_lut_maps(worldcover_path, lut)

## 5. Biophysical processing of Sentinel-2 data

Many of the Sentinel-2 based biophysical parameters were already collected during the previous notebook. Here they are saved as individual GeoTiff files and used to derive the remaining biophysical parameters, such as fraction of vegetation which is green, canopy height or reflectance of the canopy. 

In [ ]:
biopar_biophysical_params(s2_path, worldcover_path) 

## 3. Sharpen Sentinel-3 Land Surface Temperature

Sentinel-3 Lands Surace Temperature (LST) is acquired with around 1 km spatial resolution. In this step, data fusion with high-resolution Sentinel-2 reflectance, DEM and DEM-based illumination conditions is performed to derive a high-resolution representation of the LST. 

**Note:** the `n_jobs` parameter is set to 2 to speed up the processing. However, for larger AOIs this can lead to too much memory consumption and the notebook crashing. In this case, set this parameter to 1.

In [ ]:
s2_refl_file = list(date_data_dir.glob("*_REFL.tif"))[0]
dem_path = aoi_data_dir / "cdem.tif"
for lst_file in date_data_dir.glob("*_LST.tif"):
    datetime_utc = datetime.strptime(re.search(r"_(\d{8}T\d{6})_", lst_file.stem).group(1), "%Y%m%dT%H%M%S")
    mask_file = Path(str(lst_file).replace("_LST.tif", "_mask.tif"))
    output_file = Path(str(lst_file).replace("_LST.tif", "_LST_sharpened.tif"))

    sharpened_data = sharpen_lst(
        high_res_optical_path=s2_refl_file,
        high_res_dem_path=dem_path,
        low_res_lst=lst_file,
        low_res_lst_mask=mask_file,
        mask_values=[1],
        datetime_utc=datetime_utc, 
        cv_homogeneity_threshold=0,
        moving_window_size=20,
        disaggregating_temperature=True,
        n_jobs=2,
        n_estimators=30,
        max_samples=0.8,
        max_features=0.8,
        output_path=output_file
    )

# Resample VZA to Sentinel-2 grid
for vza_file in date_data_dir.glob("*_VZA.tif"):
    raster_utils.resample_to_template(vza_file, vza_file, s2_refl_file)

#### Visualize the sharpened LST

In [ ]:
map = visualization.show_raster_map(
    output_file,
    cmap="inferno"
)
map

## 4. Collect and prepare Meteorological Data
Meteorological data is the only data that is not collected from CDSE using the openEO interface. Instead it come from the [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/) and [Atmosphere Data Store](https://cds.climate.copernicus.eu/). To access data from these two sources you need an API key as described in the documentation:
* [CDS User Guide](https://cds.climate.copernicus.eu/how-to-api) 

Once you create the keys save them in ```key.adsapirc``` and ```key.cdsapirc``` files, looking like the examples below, and upload to `./mystorage` if running on CDSE Jupyterhub:

``` bash
key.adsapirc
url: https://ads.atmosphere.copernicus.eu/api
key: <api_key>
```
and
```` bash
key.cdsapirc
url: https://cds.climate.copernicus.eu/api
key: <api_key>
````

The cell below downloads the data, performs DEM-based corrections to some parameters and resamples to Sentinel-2 grid.

In [ ]:
template_file = list(date_data_dir.glob("*_LAI.tif"))[0]
for lst_file in date_data_dir.glob("*_LST.tif"):
    sentinel3_acq_datetime_str = re.search("_(\d{8}T\d{6})_", lst_file.stem).group(1)
    sentinel3_acq_datetime = datetime.strptime(sentinel3_acq_datetime_str, "%Y%m%dT%H%M%S")
    meteo_output_path = get_meteo_data(
        datetime = sentinel3_acq_datetime,
        bbox = bbox,
        dem_path = aoi_data_dir / "cdem_300.tif",
        template_path = template_file,
        data_dir=lst_file.parent,
        cds_credentials_file=".cdsapirc",
        ads_credentials_file=".adsapirc",
    )